# 01 – Feature Exploration
Smoke-test all three branches on the 13-sample dataset. Visualise:
- DINOv2 self-attention maps
- SRM noise residuals
- Physical feature PCA

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from pathlib import Path
from PIL import Image
from tqdm.notebook import tqdm

from src.utils.config import load_config
from src.utils.seed import seed_everything
from src.transforms import get_val_transform
from src.models.physical_features import extract_physical_features, _SRM_KERNELS_3x3
import cv2

ROOT = Path('..')
cfg = load_config(ROOT / 'config.yaml')
seed_everything(cfg.data.seed)

df = pd.read_csv(ROOT / cfg.data.train_csv)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}  | n_samples: {len(df)}')

## Physical Artifact Features

In [ ]:
# Extract physical features for all 13 samples
phys_feats = []
for _, row in tqdm(df.iterrows(), total=len(df)):
    img = Image.open(ROOT / row['image_path']).convert('RGB')
    f = extract_physical_features(img, wavelet=cfg.physical.wavelet,
                                  wavelet_level=cfg.physical.wavelet_level,
                                  color_bins=cfg.physical.color_bins)
    phys_feats.append(f)

phys_feats = np.stack(phys_feats)
print(f'Physical feature dim: {phys_feats.shape[1]}')

In [ ]:
# PCA of physical features (2D) coloured by label
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

labels = df['label'].values
X_scaled = StandardScaler().fit_transform(phys_feats)
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

fig, ax = plt.subplots(figsize=(7, 5))
scatter = ax.scatter(X_pca[:, 0], X_pca[:, 1],
                     c=labels, cmap='RdYlGn_r', s=100, edgecolors='k', linewidths=0.8)
plt.colorbar(scatter, ax=ax, label='label (1=fraud)')
for i, row in df.iterrows():
    ax.annotate(row['type'], (X_pca[i,0], X_pca[i,1]), fontsize=7, alpha=0.7)
ax.set_title('Physical features – PCA (2D)'); ax.set_xlabel('PC1'); ax.set_ylabel('PC2')
plt.tight_layout(); plt.show()

In [ ]:
# Visualise SRM noise residuals for one genuine and one fraud sample
genuine_row = df[df['label'] == 0].iloc[0]
fraud_row   = df[df['label'] == 1].iloc[0]

def show_srm(row, title):
    img = np.array(Image.open(ROOT / row['image_path']).convert('RGB'))
    # Resize for display
    img = cv2.resize(img, (256, 256))
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY).astype(np.float32)
    residuals = [cv2.filter2D(gray, -1, k) for k in _SRM_KERNELS_3x3]
    return img, residuals

fig, axes = plt.subplots(2, 5, figsize=(16, 7))
for row_i, (row, label_str) in enumerate([(genuine_row, 'Genuine'), (fraud_row, 'Fraud')]):
    img, residuals = show_srm(row, label_str)
    axes[row_i][0].imshow(img); axes[row_i][0].set_title(f'{label_str}\n{row["type"]}', fontsize=9)
    axes[row_i][0].axis('off')
    for j, res in enumerate(residuals[:4]):
        axes[row_i][j+1].imshow(np.abs(res), cmap='hot', vmin=0, vmax=50)
        axes[row_i][j+1].set_title(f'SRM filter {j+1}', fontsize=8)
        axes[row_i][j+1].axis('off')

plt.suptitle('SRM Noise Residuals (genuine vs fraud)', fontsize=12)
plt.tight_layout(); plt.show()

## DINOv2 Self-Attention Maps

In [ ]:
# Load DINOv2 and visualise attention maps for genuine vs fraud
from transformers import AutoModel
import torchvision.transforms.functional as TF

transform = get_val_transform(image_size=448)  # patch_size=14, so 448/14=32 patches/side

dino = AutoModel.from_pretrained(
    cfg.dinov2.model_name,
    output_attentions=True,
).to(device).eval()
print('DINOv2 loaded')

In [ ]:
def get_attention_map(img_path, size=448):
    img = Image.open(ROOT / img_path).convert('RGB')
    tensor = transform(img).unsqueeze(0).to(device)
    with torch.no_grad():
        out = dino(pixel_values=tensor, output_attentions=True)
    # Last layer attention: (1, n_heads, n_patches+1, n_patches+1)
    attn = out.attentions[-1][0]          # (n_heads, N+1, N+1)
    attn_cls = attn[:, 0, 1:]             # attention from CLS to all patches: (n_heads, N)
    attn_mean = attn_cls.mean(0).cpu().numpy()  # (N,)
    patch_side = int(size // dino.config.patch_size)
    attn_map = attn_mean.reshape(patch_side, patch_side)
    # Normalise
    attn_map = (attn_map - attn_map.min()) / (attn_map.max() - attn_map.min() + 1e-8)
    return img, attn_map

fig, axes = plt.subplots(2, 3, figsize=(13, 9))
import matplotlib.cm as cm

for row_i, (row, label_str) in enumerate([(genuine_row, 'Genuine'), (fraud_row, 'Fraud')]):
    img, attn = get_attention_map(row['image_path'])
    img_resized = img.resize((448, 448))
    attn_overlay = Image.fromarray((cm.jet(attn)[:, :, :3] * 255).astype(np.uint8)).resize((448, 448))
    blend = Image.blend(img_resized.convert('RGBA'), attn_overlay.convert('RGBA'), alpha=0.5).convert('RGB')

    axes[row_i][0].imshow(img_resized); axes[row_i][0].set_title(f'{label_str}: {row["type"]}\nOriginal', fontsize=9)
    axes[row_i][1].imshow(attn, cmap='jet'); axes[row_i][1].set_title('Attention map', fontsize=9)
    axes[row_i][2].imshow(blend); axes[row_i][2].set_title('Overlay', fontsize=9)
    for ax in axes[row_i]: ax.axis('off')

plt.suptitle('DINOv2 Self-Attention (last layer, CLS token)', fontsize=12)
plt.tight_layout(); plt.show()

## DINOv2 Embeddings PCA

In [ ]:
from src.models.global_backbone import DINOv2Backbone

dino_model = DINOv2Backbone(
    model_name=cfg.dinov2.model_name,
    frozen=True,
    use_cls=True, use_patch_mean=True, use_patch_stats=False,
).to(device).eval()

all_feats = []
for _, row in tqdm(df.iterrows(), total=len(df)):
    img = Image.open(ROOT / row['image_path']).convert('RGB')
    tensor = transform(img).unsqueeze(0).to(device)
    with torch.no_grad():
        feat = dino_model(tensor).cpu().numpy()
    all_feats.append(feat[0])

dino_feats = np.stack(all_feats)
print(f'DINOv2 feature dim: {dino_feats.shape[1]}')

X_dino = StandardScaler().fit_transform(dino_feats)
X_pca_dino = PCA(n_components=2).fit_transform(X_dino)

fig, ax = plt.subplots(figsize=(7, 5))
scatter = ax.scatter(X_pca_dino[:, 0], X_pca_dino[:, 1],
                     c=labels, cmap='RdYlGn_r', s=100, edgecolors='k')
plt.colorbar(scatter, ax=ax, label='label (1=fraud)')
for i, row in df.iterrows():
    ax.annotate(row['type'], (X_pca_dino[i,0], X_pca_dino[i,1]), fontsize=7, alpha=0.7)
ax.set_title('DINOv2 CLS+patch embeddings – PCA (2D)')
plt.tight_layout(); plt.show()